# Notebook 16 - Dual-View Adapter Inference Ablation

Runs the same adapter on both whole image and router/SAM ROI crop, then selects the ROI prediction only when it clears the confidence margin.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            repo_root = candidate.resolve()
            os.chdir(repo_root)
            try:
                subprocess.run(['git', 'pull', '--ff-only'], check=True)
            except Exception as exc:
                print(f'[BOOT] git pull skipped/failed: {exc}', flush=True)
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    os.chdir(CLONE_TARGET)
    return CLONE_TARGET

ROOT = _ensure_aads_repo_on_path()
from scripts.colab_notebook_bootstrap_helpers import setup_notebook_environment
ROOT = setup_notebook_environment(install_requirements=True, print_tokens=True)
print(f'[BOOT] ROOT={ROOT}')


In [ ]:
from pathlib import Path
import importlib
import scripts.colab_roi_ablation as roi_ablation

roi_ablation = importlib.reload(roi_ablation)
commit_and_push_ablation_results = roi_ablation.commit_and_push_ablation_results
run_dual_view_inference_folder = roi_ablation.run_dual_view_inference_folder

ABLATION_NAME = 'dual_view_inference'
DATASET_KEY = 'tomato__fruit'
IMAGE_DIR = ROOT / 'data' / 'prepared_runtime_datasets' / DATASET_KEY / 'test'
ADAPTER_ROOT = ROOT / 'runs' / 'tomato' / 'fruit' / 'tomato_fruit_2026-05-26_10-34-27' / 'outputs' / 'colab_notebook_training'
ADAPTER_CROP = 'tomato'
ADAPTER_PART = 'fruit'
OUTPUT_DIR = ROOT / 'docs' / 'ablation_results' / ABLATION_NAME
CONFIG_ENV = 'colab'
DEVICE = 'cuda'
ROI_CONFIDENCE_MARGIN = 0.10

print(f'[CONFIG] ablation={ABLATION_NAME} image_dir={IMAGE_DIR} output={OUTPUT_DIR}')


In [ ]:
if not Path(IMAGE_DIR).is_dir():
    raise FileNotFoundError(f'IMAGE_DIR not found: {IMAGE_DIR}')
if not Path(ADAPTER_ROOT).is_dir():
    raise FileNotFoundError(f'ADAPTER_ROOT not found: {ADAPTER_ROOT}')

report = run_dual_view_inference_folder(
    IMAGE_DIR,
    output_dir=OUTPUT_DIR,
    adapter_crop=ADAPTER_CROP,
    adapter_part=ADAPTER_PART,
    config_env=CONFIG_ENV,
    device=DEVICE,
    adapter_root=ADAPTER_ROOT,
    roi_confidence_margin=ROI_CONFIDENCE_MARGIN,
)
git_result = commit_and_push_ablation_results(
    OUTPUT_DIR,
    repo_root=ROOT,
    message='Add dual-view ROI inference ablation results',
)
{'summary': report['summary'], 'git': git_result}
